# 단권·3권 책 제목 인식 테스트 (셀 단위)
`eval/books/` 의 사진 8장으로 **우리 모델(E)** 의 권수·제목 인식을 확인합니다.
- 정답: 사용자가 알려 준 실제 제목 · 비교용: Claude가 사진을 보고 읽은 판독(`eval/books_gt.json`) — **파이프라인 코드에는 LLM을 쓰지 않습니다.**
- 순서: 1 → 2 실행 후, 3번에서 사진 이름을 바꿔 가며 한 장씩 확인 → 5번에서 전체 표

## 1. 준비 (처음 한 번)

In [ ]:
import os, sys, json, re, time, unicodedata, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

ROOT = Path.cwd()                                   # shelf_ocr_test 폴더에서 실행
sys.path.insert(0, str(ROOT / "src"))
import importlib, matcher, shelfkit; importlib.reload(matcher); importlib.reload(shelfkit)
from shelfkit import *
from matcher import to_jamo
from rapidfuzz.distance import Indel

for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"]:
    if any(f in x.name for x in matplotlib.font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f; break
pd.set_option("display.max_colwidth", 60)

EV = ROOT / "eval"
GT = json.load(open(EV / "books_gt.json", encoding="utf-8"))
BOOKS = GT["_books"]
PHOTOS = [k for k in GT if not k.startswith("_")]

def norm(s): return re.sub(r"[^0-9a-z가-힣]", "", unicodedata.normalize("NFC", s).lower())
def sim(x, y):
    x, y = norm(x), norm(y)
    return 100 * Indel.normalized_similarity(to_jamo(x), to_jamo(y)) if x and y else 0.0
def score(titles, keys):
    """정답 책마다 별칭과 자모 유사도 90 이상인 제목이 있으면 정답 (1:1 짝짓기, 괄호 속 설명 무시)"""
    titles = [re.sub(r"\s*\(.*?\)", "", t) for t in titles]
    pairs = sorted(((max(sim(t, a) for a in BOOKS[k]["aliases"]), i, j) for i, t in enumerate(titles) for j, k in enumerate(keys)), reverse=True)
    ut, ub, hit = set(), set(), 0
    for s, i, j in pairs:
        if s >= 90 and i not in ut and j not in ub: ut.add(i); ub.add(j); hit += 1
    return hit

print("테스트 사진:", PHOTOS)

## 2. 모델 불러오기 (LLM 없음)

In [ ]:
catalog = load_catalog(ROOT); matcher_ = TitleMatcher(catalog)
UNIFIED = ROOT / "models" / "books_obb_best.pt"       # tools/train_detector.py 로 학습한 책등·표지 통합 검출기
DETECTOR = "unified" if UNIFIED.exists() else "rules"  # "rules" = 기존(책등 모델 + YOLO11x 6회 + 규칙)으로 비교하려면 직접 지정
if DETECTOR == "unified":
    detector = UnifiedBookDetector(UNIFIED, imgsz=960, conf=0.35)
else:
    detector = BookDetector(ROOT / "models" / "spine_obb_best.pt", ROOT / "models" / "yolo11x.pt", imgsz=960, conf=0.45, iou=0.5)
print("검출기:", DETECTOR)
reader = TitleRuleReader(PaddleRecEngine("korean_PP-OCRv5_mobile_rec", "ko"), PaddleRecEngine("en_PP-OCRv5_mobile_rec", "en"), detector.textdet,
                         label="E · 제목 규칙")
print(f"카탈로그 {len(catalog)}권 · 사용 모델: YOLO11n-OBB(책등), YOLO11x COCO book(표지), {', '.join(reader.models)}")
RESULTS = {}   # 사진별 결과 (5번 표에 사용)

def run_one(name, show=True):
    img = cv2.imdecode(np.fromfile(str(EV / "books" / name), np.uint8), cv2.IMREAD_COLOR)
    t0 = time.perf_counter()
    det = detect_image(img, detector)                    # 검출: 1600px
    r = run_image(img, detector, reader, matcher_, det=det)   # 글자: 고해상도(최대 3000px)
    ms = (time.perf_counter() - t0) * 1000
    g = GT[name]
    titles = [b["final"] for b in r["books"] if b["final"] != "Unknown"]
    hit = score(titles, g["books"])
    RESULTS[name] = {"count": len(r["books"]), "titles": titles, "ms": ms, "ms_det": det["t_det"] * 1000,
                     "books": [{"kind": b["kind"], "final": b["final"], "source": b["source"], "raw": b.get("raw") or "", "view": b.get("view", "")} for b in r["books"]]}
    if show:
        vis = draw_final(r["img"], r["books"], reader.name)
        plt.figure(figsize=(24, 8)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off")
        plt.title(f"{name} · {g['label']}", fontsize=14); plt.show()
        print(f"실제: {g['count']}권 · {' / '.join(BOOKS[k]['truth'] for k in g['books'])}")
        print(f"모델: {len(r['books'])}권 {'✅' if len(r['books']) in g['count_ok'] else '❌'} · 제목 정답 {hit}/{len(g['books'])} · "
              f"{ms / 1000:.1f}초 (검출 {det['t_det']:.1f}초)")
        display(pd.DataFrame([dict(번호=i + 1, 종류="표지" if b["kind"] == "cover" else "책등", 인식된제목=b["final"], 근거=b["source"],
                                   OCR원문=b.get("raw") or "", 판독=b.get("view", "")) for i, b in enumerate(r["books"])]))
    return r

## 3. 사진 한 장씩 확인
`PHOTO` 를 바꿔 가며 실행하세요. ① 책 검출 ② 글자 박스(자홍 = 제목으로 쓴 큰 글씨) ③ 인식 제목 + 아래 표(OCR 원문, 판독 방향)
- IMG_1587 Embedded 표지 · 1588 Embedded 책등 · 1589 회로이론 표지(90°) · 1590 회로이론 책등
- IMG_1591 Kreyszig 표지 · 1592 Kreyszig 책등 · 1593 3권(눕힘+세움) · 1594 3권(쌓음)

In [ ]:
PHOTO = "IMG_1592.jpg"
r = run_one(PHOTO)

## 3-1. Step 1 확인 — 책 검출이 어떻게 되었나 (4단)
3번에서 돌린 사진의 검출 과정을 그대로 보여 줍니다.
- ① **책등 모델 원본**(자홍): 걸러내기 전. 책 안의 글줄·무늬를 책등으로 잡았는지 확인
- ② **COCO book 후보**: 숫자 = 지지도·가로세로비. **빨강 얇은 선 = 비율이 커서(>2.4) 버려진 상자** → 눕힌 책이 여기서 버려지는지 확인
- ③ **규칙 통과 후보**: 초록 = 표지, 파랑 = 책등
- ④ **최종 검출**

In [ ]:
vis = draw_debug_step1(r["img"], r["stages"], r["books"])
plt.figure(figsize=(22, 22 * vis.shape[0] / vis.shape[1])); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
st = r["stages"]
print(f"책등 모델 원본 {len(st['raw_spines'])}개 → 검증 통과 {len(st['spines'])}개 · COCO 후보 {sum(c['support'] >= 0.02 for c in st['fused'])}개 → 표지 {len(st['covers'])}개 · 표지 위주 사진: {st['cover_scene']}")

## 3-2. Step 2 확인 — 책마다 어디를 읽었나
검출된 책마다 잘라낸 영역 위에 글자 상자를 그립니다. **자홍 = 제목으로 쓴 큰 글씨**, 하늘 = 읽었지만 제목에서 뺀 글씨. 아래에 최종 제목과 OCR 원문.

In [ ]:
n = len(r["books"])
if n == 0:
    print("검출된 책이 없습니다 → 3-1의 ②에서 눕힌 책이 빨간 선(비율 초과)으로 버려졌는지 확인하세요.")
else:
    cols = min(3, n); rows_ = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(7 * cols, 5 * rows_)); axes = np.atleast_1d(axes).ravel()
    for ax, (i, b) in zip(axes, enumerate(r["books"])):
        x1, y1, x2, y2 = [int(v) for v in b["bbox"]]; pad = 10
        x1, y1 = max(0, x1 - pad), max(0, y1 - pad); x2, y2 = min(r["img"].shape[1], x2 + pad), min(r["img"].shape[0], y2 + pad)
        sub = r["img"][y1:y2, x1:x2].copy()
        for tb in b["text_boxes"]:
            if tb["selected"]:
                cv2.polylines(sub, [np.int32(np.asarray(tb["poly"]) - [x1, y1])], True, (255, 0, 255) if tb["used"] else (230, 180, 0), 2 if tb["used"] else 1)
        ax.imshow(cv2.cvtColor(sub, cv2.COLOR_BGR2RGB)); ax.axis("off")
        ax.set_title(f"{i + 1}. [{'표지' if b['kind'] == 'cover' else '책등'}] {b['final']} ({b['source']})\nOCR: {(b.get('raw') or '')[:40]}\n{b.get('view', '')}", fontsize=10)
    for ax in axes[n:]: ax.axis("off")
    plt.tight_layout(); plt.show()

## 4. (선택) 책 한 권 자세히 보기
3번을 실행한 사진에서 `BOOK_NO` 번 책의 크롭과 읽은 글자 상자를 확인합니다. 제목이 틀리면 여기서 원인을 봅니다.

In [ ]:
BOOK_NO = 1
b = r["books"][BOOK_NO - 1]
img_full = cv2.imdecode(np.fromfile(str(EV / "books" / PHOTO), np.uint8), cv2.IMREAD_COLOR)
k = min(1.0, 3000 / max(img_full.shape[:2])); img_full = cv2.resize(img_full, None, fx=k, fy=k) if k < 1 else img_full
s = img_full.shape[1] / r["img"].shape[1]
from shelfkit import _scale_book
crop, _ = crop_book(img_full, _scale_book(b, s))
plt.figure(figsize=(14, 4)); plt.imshow(cv2.cvtColor(crop if crop.shape[0] < crop.shape[1] else cv2.rotate(crop, cv2.ROTATE_90_COUNTERCLOCKWISE), cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"{BOOK_NO}번 책 크롭 ({'표지' if b['kind'] == 'cover' else '책등'}, {crop.shape[1]}×{crop.shape[0]}px)"); plt.show()
print("최종 제목:", b["final"], f"[{b['source']}]", "· 판독:", b.get("view"))
display(pd.DataFrame([dict(제목사용="✔" if tb["used"] else "", 읽은글자=tb["text"], 글자크기=tb["size"]) for tb in b["text_boxes"] if tb["selected"]]))

## 5. 전체 8장 실행 → 결과 표 (실제 / Claude / 우리 모델 PC / 모바일)

In [ ]:
SHOW_IMAGES = False                       # True면 사진마다 3단 이미지도 출력
for name in PHOTOS:
    if name not in RESULTS:
        run_one(name, show=SHOW_IMAGES); print(f"{name}: {RESULTS[name]['count']}권 · {RESULTS[name]['titles']} · {RESULTS[name]['ms'] / 1000:.1f}s")
json.dump(RESULTS, open(EV / "res_books_pc.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

MOBILE = {}   # 모바일 웹 데모 결과: {"IMG_1587.jpg": {"count": 1, "titles": [...], "ms": 1234}, ...} 를 붙여 넣거나 json.load
rows, tot = [], {k: [0, 0] for k in ["c_cnt", "c_ttl", "p_cnt", "p_ttl", "m_cnt", "m_ttl"]}
for name in PHOTOS:
    g = GT[name]; keys = g["books"]; row = {"사진": name, "구성": g["label"], "실제 권수": g["count"], "실제 제목": " / ".join(BOOKS[k]["truth"] for k in keys)}
    for tag, key, src in [("Claude", "c", g["claude"]), ("PC", "p", RESULTS.get(name)), ("모바일", "m", MOBILE.get(name))]:
        if not src: row.update({f"{tag} 권수": "-", f"{tag} 제목": "-", f"{tag} 정답": "-", f"{tag} 초": "-"}); continue
        hit = score(src["titles"], keys); ok = src["count"] in g["count_ok"]
        tot[f"{key}_cnt"][0] += ok; tot[f"{key}_cnt"][1] += 1; tot[f"{key}_ttl"][0] += hit; tot[f"{key}_ttl"][1] += len(keys)
        row.update({f"{tag} 권수": f"{src['count']}{'' if ok else ' ✗'}", f"{tag} 제목": " / ".join(src["titles"]) or "Unknown",
                    f"{tag} 정답": f"{hit}/{len(keys)}", f"{tag} 초": f"{src['ms'] / 1000:.2f}" if "ms" in src else "참고용"})
    rows.append(row)
pct = lambda x: f"{100 * x[0] / x[1]:.0f}% ({x[0]}/{x[1]})" if x[1] else "-"
avg = lambda d: f"{np.mean([v['ms'] for v in d.values() if 'ms' in v]) / 1000:.2f}" if d else "-"
summary = pd.DataFrame([
    {"구분": "실제 (정답)", "권수 정답률": "100%", "제목 정답률": "100%", "사진당 평균 시간(초)": "-"},
    {"구분": "Claude 판독 (LLM, 비교용)", "권수 정답률": pct(tot["c_cnt"]), "제목 정답률": pct(tot["c_ttl"]), "사진당 평균 시간(초)": "참고용"},
    {"구분": "우리 모델 E · PC", "권수 정답률": pct(tot["p_cnt"]), "제목 정답률": pct(tot["p_ttl"]), "사진당 평균 시간(초)": avg(RESULTS)},
    {"구분": "우리 모델 · 모바일 (Edge)", "권수 정답률": pct(tot["m_cnt"]), "제목 정답률": pct(tot["m_ttl"]), "사진당 평균 시간(초)": avg(MOBILE)}])
display(summary); display(pd.DataFrame(rows))
pd.DataFrame(rows).to_csv(EV / "books_result.csv", index=False, encoding="utf-8-sig")
print("저장:", EV / "res_books_pc.json", "·", EV / "books_result.csv")